In [1]:
# =============================
# INSTALL LIBRARIES
# =============================
!pip install -q transformers datasets jiwer accelerate huggingface_hub
!pip install matplotlib

# =============================
# IMPORTS
# =============================
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator
)
from huggingface_hub import login
import matplotlib.pyplot as plt

# =============================
# LOGIN
# =============================

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token  = user_secrets.get_secret("HF_TOKEN")

login(hf_token)

# =============================
# LOAD YOUR SAVED MODEL (v1)
# =============================
model_id = "hasindu-k/sinhala-handwritten-notes"

processor = TrOCRProcessor.from_pretrained(model_id)
model = VisionEncoderDecoderModel.from_pretrained(model_id)

# =============================
# DATA PATH
# =============================
DATA_DIR = "/kaggle/input/datasets/hasindukoshitha/sinhala-ocr-training-data/train_images/"
CSV_PATH = "/kaggle/input/datasets/hasindukoshitha/sinhala-ocr-training-data/metadata.csv"

class SinhalaDataset(torch.utils.data.Dataset):
    def __init__(self, df, processor):
        self.df = df
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        file_name = self.df.iloc[idx]['file_name']
        text = self.df.iloc[idx]['text']

        # Ensure pathing is correct for Kaggle
        img_path = os.path.join(DATA_DIR, file_name)
        image = Image.open(img_path).convert("RGB")

        # Process image only
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()

        # Tokenize text separately with fixed padding
        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=64,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze()

        # Replace pad token with -100
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

# =============================
# LOAD DATA
# =============================
df = pd.read_csv(CSV_PATH)
dataset = SinhalaDataset(df, processor)

# =============================
# TRAINING SETTINGS
# =============================
train_loss_values = []

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/results",
    per_device_train_batch_size=8,
    num_train_epochs=10,
    learning_rate=2e-5,  # smaller LR for continued training
    fp16=True,
    save_strategy="epoch",
    logging_steps=20,
    eval_strategy="no",
    report_to="tensorboard",  # Enable TensorBoard for tracking
)

# Define a custom trainer to capture the loss values and plot them
class CustomTrainer(Seq2SeqTrainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_values = []

    def log(self, logs, *args, **kwargs):
        super().log(logs, *args, **kwargs)
        if 'loss' in logs:
            self.loss_values.append(logs['loss'])

    def plot_training_loss(self):
        plt.figure(figsize=(10, 5))
        plt.plot(self.loss_values)
        plt.xlabel('Steps (Logging Frequency)')
        plt.ylabel('Loss')
        plt.title('Sinhala OCR Training Loss')
        plt.show()

# Create trainer instance
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=default_data_collator
)

# =============================
# START TRAINING
# =============================
trainer.train()

# Plot the training loss graph after training
trainer.plot_training_loss()

# =============================
# SAVE MODEL (v2)
# =============================
save_path = "/kaggle/working/sinhala_model_v2"
trainer.save_model(save_path)
processor.save_pretrained(save_path)

print("✅ Continued training completed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.4 MB/s eta 0:00:0000:0100:01


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: [enforce fail at inline_container.cc:664] . unexpected pos 692627136 vs 692627028

In [4]:
# Save the model to a specific path
save_path = "/kaggle/working/sinhala_model_v2"
trainer.save_model(save_path)
processor.save_pretrained(save_path)

print(f"✅ Model saved at {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved at /kaggle/working/sinhala_model_v2


In [5]:
# =============================
# PUSH TO HUGGING FACE
# =============================

repo_name = "sinhala-handwritten-notes-v2"

trainer.push_to_hub(repo_name)
processor.push_to_hub(repo_name)

print("🚀 Model pushed successfully to Hugging Face!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

🚀 Model pushed successfully to Hugging Face!


In [ ]:
import transformers
print(transformers.__version__)

In [3]:
import shutil

# Example: Remove unnecessary checkpoints
shutil.rmtree('/kaggle/working/results/checkpoint-132')
shutil.rmtree('/kaggle/working/results/checkpoint-165')